# Compact stochastic Neural-DTB oscillatory game

This notebook contains one complete stochastic experiment for one parameter choice. It derives the drift equilibria and stationary density, samples uniform initial particles, computes an Euler--Maruyama SDE reference, evolves the same initial distribution with the Neural-DTB probability flow, and makes only three diagnostic figures.

The notebook imports the existing DTB core, but it does not call the repository's sweep runner or generate tables and output folders. In Google Colab, first choose **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom.

In [ ]:
# Colab setup
import os, pathlib, subprocess, sys

REPO_DIR = pathlib.Path("/content/dtb-colab-experiments")
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "-q", "--depth", "1",
        "--branch", "codex/game-dynamics-dtb",
        "https://github.com/sun-mengwei/dtb-colab-experiments.git",
        str(REPO_DIR),
    ], check=True)
os.chdir(REPO_DIR / "dtb_game_dynamics_unnormalized")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## Game and numerical setup

For $x=(x_1,x_2)$, the potential and game velocity are

$$\Phi(x)=-\frac{\lambda}{2}(x_1^2+x_2^2)-\frac{\gamma}{2}(x_1-x_2)^2+\frac{\varepsilon}{\omega}[\cos(\omega x_1)+\cos(\omega x_2)],$$

$$b(x)=\nabla\Phi(x)=\begin{bmatrix}-\lambda x_1-\gamma(x_1-x_2)-\varepsilon\sin(\omega x_1)\\-\lambda x_2-\gamma(x_2-x_1)-\varepsilon\sin(\omega x_2)\end{bmatrix}.$$

The stochastic dynamics and their Fokker--Planck probability-flow velocity are

$$dX_t=b(X_t)\,dt+\sigma\,dW_t, \qquad v(x,t)=b(x)-\frac{\sigma^2}{2}\nabla\log\rho(x,t).$$

DTB uses the diffusion matrix $D=\sigma^2I$ and transports the score $q=\nabla\log\rho$. Euler--Maruyama and DTB start from the same particles sampled uniformly from $[-1,1]^2$. The initial score is zero in the box interior; the uniform-density boundary is a measure-zero nonsmooth set under this particle convention. Their particle paths should not match one by one; the comparison is between their distributions at the same times.

## Analytical equilibrium equations

An equilibrium $x^*=(x_1^*,x_2^*)$ satisfies $b(x^*)=0$, or

$$ (\lambda+\gamma)x_1^*-\gamma x_2^*+\varepsilon\sin(\omega x_1^*)=0, \qquad (\lambda+\gamma)x_2^*-\gamma x_1^*+\varepsilon\sin(\omega x_2^*)=0. $$

Use the mean and difference coordinates

$$m=\frac{x_1+x_2}{2}, \qquad d=\frac{x_1-x_2}{2}.$$

Adding and subtracting the two equilibrium equations gives the exact characterization

$$\boxed{\lambda m+\varepsilon\sin(\omega m)\cos(\omega d)=0},$$

$$\boxed{(\lambda+2\gamma)d+\varepsilon\cos(\omega m)\sin(\omega d)=0}.$$

Thus the symmetric equilibria $(s,s)$ are the roots of

$$\lambda s+\varepsilon\sin(\omega s)=0,$$

and the antisymmetric equilibria $(s,-s)$ are the roots of

$$(\lambda+2\gamma)s+\varepsilon\sin(\omega s)=0.$$

Because these are transcendental equations, nonzero roots generally have no elementary closed form; the two boxed equations are the analytical solution condition, and individual nonzero roots must be evaluated numerically. The origin is always an exact equilibrium.

The game Jacobian is

$$J_b(x)=\begin{bmatrix}-\lambda-\gamma-\varepsilon\omega\cos(\omega x_1)&\gamma\\\gamma&-\lambda-\gamma-\varepsilon\omega\cos(\omega x_2)\end{bmatrix}.$$

Since $b=\nabla\Phi$, an isolated equilibrium is locally asymptotically stable when this symmetric matrix is negative definite. At the origin its eigenvalues are

$$\mu_{\mathrm{mean}}=-\lambda-\varepsilon\omega, \qquad \mu_{\mathrm{difference}}=-\lambda-2\gamma-\varepsilon\omega.$$

Therefore the origin is stable exactly when both eigenvalues are negative. For the parameters below they are approximately $-6.7832$ and $-7.1832$, so at least one stable drift equilibrium exists.

With nonzero isotropic noise, individual SDE particles do not remain at these equilibria. The corresponding stationary density is

$$\rho_\infty(x)=Z^{-1}\exp\!\left(\frac{2\Phi(x)}{\sigma^2}\right),$$

so the stable wells of $\Phi$ become modes of a smooth probability distribution.

In [ ]:
import math
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import torch

from game_dtb.algorithm import DTBConfig, NeuralDTBGameDynamics
from game_dtb.models import TangentMLP
from game_dtb.runner import DTBProgressReporter, simulate_euler_maruyama
from game_dtb.state import ParticleState

# One game only. Change these values later to study another case.
LAMBDA = 0.5
GAMMA = 0.2
EPSILON = 0.5
OMEGA = 4.0 * math.pi
NOISE_STD = 0.1

PARTICLES = 128
STEPS = 1000
STEP_SIZE = 0.001       # 1000 steps keep the final time T = 1
SEED = 2026
MODEL_SEED = 91
WIDTH = 12
DEPTH = 2
BASIS_SIZE = 24
SVD_RTOL = 1e-3

# Periodically fit the NN to the accumulated DTB tangent target.
# Set REFIT_INTERVAL = 0 to recover the fixed-network version.
REFIT_INTERVAL = 100
REFIT_OPTIMIZER_STEPS = 300
REFIT_LEARNING_RATE = 1e-3
REFIT_SAMPLES = 1024
REFIT_BATCH_SIZE = 256
REFIT_TEST_SAMPLES = 512

DTYPE = torch.float32
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.set_float32_matmul_precision("high")

pprint({
    "lambda": LAMBDA, "gamma": GAMMA, "epsilon": EPSILON,
    "omega": f"{OMEGA / math.pi:g} pi",
    "noise_std": NOISE_STD, "diffusion_diagonal": NOISE_STD ** 2,
    "particles": PARTICLES, "steps": STEPS,
    "step_size": STEP_SIZE, "final_time": STEPS * STEP_SIZE,
    "network": f"TangentMLP(width={WIDTH}, depth={DEPTH})",
    "basis_size": BASIS_SIZE, "svd_rtol": SVD_RTOL,
    "refit_interval": REFIT_INTERVAL,
    "planned_refits": STEPS // REFIT_INTERVAL if REFIT_INTERVAL else 0,
    "device": str(DEVICE),
})
if DEVICE.type != "cuda":
    print("GPU not detected. In Colab choose Runtime > Change runtime type > T4 GPU.")

## Functions used in this experiment

These are the complete experiment-specific functions. The imported DTB class performs the tangent projection and particle update. There are no sweep, report, dimension-check, or test-suite cells here.

In [ ]:
def potential(x):
    x1, x2 = x.unbind(dim=-1)
    return (
        -0.5 * LAMBDA * (x1.square() + x2.square())
        -0.5 * GAMMA * (x1 - x2).square()
        +(EPSILON / OMEGA) * (torch.cos(OMEGA * x1) + torch.cos(OMEGA * x2))
    )


def velocity(x):
    x1, x2 = x.unbind(dim=-1)
    v1 = -LAMBDA * x1 - GAMMA * (x1 - x2) - EPSILON * torch.sin(OMEGA * x1)
    v2 = -LAMBDA * x2 - GAMMA * (x2 - x1) - EPSILON * torch.sin(OMEGA * x2)
    return torch.stack((v1, v2), dim=-1)


def make_initial_state():
    generator = torch.Generator().manual_seed(SEED)
    labels = 2.0 * torch.rand(PARTICLES, 2, generator=generator, dtype=DTYPE) - 1.0
    labels = labels.to(DEVICE)
    return ParticleState(
        particles=labels.clone(),
        log_density=torch.full((PARTICLES,), -math.log(4.0), device=DEVICE, dtype=DTYPE),
        score=torch.zeros_like(labels),
        labels=labels,
    )


def solve_with_euler_maruyama(initial_particles):
    # Reuse the generic SDE reference implementation in game_dtb/runner.py.
    schedule = {step: step * STEP_SIZE for step in range(STEPS + 1)}
    return simulate_euler_maruyama(
        initial_particles.detach().cpu().numpy(),
        velocity,
        schedule,
        steps=STEPS,
        step_size=STEP_SIZE,
        noise_std=NOISE_STD,
        device=DEVICE,
        dtype=DTYPE,
        seed=SEED + 10_000,
    )


def solve_with_dtb(initial_state):
    torch.manual_seed(MODEL_SEED)
    model = TangentMLP(
        dim=2, width=WIDTH, depth=DEPTH, activation="tanh", dtype=DTYPE
    ).to(DEVICE)
    config = DTBConfig(
        step_size=STEP_SIZE, basis_size=BASIS_SIZE, svd_rtol=SVD_RTOL,
        jacobian_chunk_size=PARTICLES, derivative_chunk_size=32, seed=SEED,
        refit_interval=REFIT_INTERVAL,
        refit_optimizer_steps=REFIT_OPTIMIZER_STEPS,
        refit_learning_rate=REFIT_LEARNING_RATE,
        refit_batch_size=REFIT_BATCH_SIZE,
        refit_samples=REFIT_SAMPLES,
        refit_test_samples=REFIT_TEST_SAMPLES,
    )
    method = NeuralDTBGameDynamics(
        model=model, drift=velocity,
        diffusion=(NOISE_STD ** 2) * torch.eye(2, device=DEVICE, dtype=DTYPE),
        config=config,
        # With no separate sampler, refitting samples the particle-label dataset.
        reference_sampler=None,
    )

    reporter = DTBProgressReporter(
        total_steps=STEPS, step_size=STEP_SIZE,
        particle_count=PARTICLES,
        basis_size=min(BASIS_SIZE, method.parameter_count),
        device=DEVICE,
    )
    state = initial_state
    history = [state.particles.cpu().numpy().copy()]
    residuals = []
    refit_steps = []
    for step in range(STEPS):
        result = method.step(state, step)
        state = result.state
        history.append(state.particles.cpu().numpy().copy())
        residuals.append(result.diagnostics.relative_residual)
        if result.refit_performed:
            refit_steps.append(step + 1)
        reporter.update(
            completed_step=step + 1, result=result,
            refit_count=method.refit_count,
        )
    return np.stack(history), np.asarray(residuals), refit_steps


def origin_stability():
    # b(0,0)=0. Negative Jacobian eigenvalues imply local asymptotic stability.
    diagonal = -LAMBDA - GAMMA - EPSILON * OMEGA
    jacobian = torch.tensor(
        [[diagonal, GAMMA], [GAMMA, diagonal]], dtype=torch.float64
    )
    eigenvalues = torch.linalg.eigvalsh(jacobian).numpy()
    return bool(eigenvalues.max() < 0.0), eigenvalues


def plot_particle_snapshots(history, title):
    indices = np.linspace(0, STEPS, 5, dtype=int)
    fig, axes = plt.subplots(1, len(indices), figsize=(15, 3), sharex=True, sharey=True)
    for axis, index in zip(axes, indices):
        points = history[index]
        axis.scatter(points[:, 0], points[:, 1], s=10, alpha=0.65)
        axis.set_title(f"t = {index * STEP_SIZE:.2f}")
        axis.set_xlim(-1.1, 1.1)
        axis.set_ylim(-1.1, 1.1)
        axis.set_aspect("equal")
        axis.grid(alpha=0.2)
    axes[0].set_ylabel(r"$x_2$")
    for axis in axes:
        axis.set_xlabel(r"$x_1$")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Run the two stochastic representations

Euler--Maruyama simulates noisy SDE paths. DTB simulates the deterministic probability flow whose one-time marginal densities should match the SDE density when the transported score is accurate. The two methods receive identical initial particles, but only their distributions—not individual labeled paths—should be compared.

The DTB loop holds one tangent basis for each 100-step block, accumulates its update, and then refits all trainable MLP parameters to that accumulated tangent target using samples drawn from the initial particle-label dataset. This is the DTB reset/refit operation—not direct supervised fitting to $b(x)$. A 1,000-step DTB run is computationally substantial, so enable a GPU in Colab before running this cell. The shared runner reports progress, projection errors, rank, refit count, elapsed time, and ETA after every 10% of the steps.

In [ ]:
initial_state = make_initial_state()
sde_history = solve_with_euler_maruyama(initial_state.particles)
dtb_history, projection_residual, refit_steps = solve_with_dtb(initial_state)
times = np.arange(STEPS + 1) * STEP_SIZE

stable, eigenvalues = origin_stability()
print(
    f"Stable drift equilibrium exists for gamma={GAMMA:g}, lambda={LAMBDA:g}: {stable}. "
    f"Origin Jacobian eigenvalues: {eigenvalues}. "
    f"NN refits completed: {len(refit_steps)} at steps {refit_steps}."
)

## Projection residual over time

At each DTB step, this is the relative residual for the probability-flow target $v=b-\tfrac12Dq$: how well the selected neural tangent directions represent the stochastic density velocity at the current particles.

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.semilogy(times[1:], np.maximum(projection_residual, 1e-12), label="DTB residual")
for index, step in enumerate(refit_steps):
    plt.axvline(times[step], color="tab:orange", alpha=0.25,
                label="NN refit" if index == 0 else None)
plt.xlabel("time")
plt.ylabel("relative projection residual")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Euler--Maruyama SDE reference over time

In [ ]:
plot_particle_snapshots(sde_history, "Euler--Maruyama SDE particles")

## Neural-DTB probability-flow solution over time

In [ ]:
plot_particle_snapshots(dtb_history, "Neural-DTB probability-flow particles")